## Implementing the BPE algorithm
The aim of this exercise is to implement the BPE tokenization algorithm. As a reminder, the principle consists in gathering the words or “tokens” that appear the most times in succession.

For example, if we consider the corpus containing the words in the following table (with the number of occurrences of each word):

| words | occurrence |
|------|-----------|
| voting | 2 |
| vote | 3 |
| slow | 1 |
| slowly | 2 |


And if the initial “tokens” are the letters of the alphabet, then the prefix “vo” will initially be added to the list of sub-words (tokens), cause the bigram "v" "o" occurs 5 times (2 + 3).

The steps involved in implementing the BPE algorithm are as follows:
1. Download a text corpus (here a wikipedia page)
2. Cut the text into words (using the “space” and “ponctuation” characters) and count the number of occurrences of each word.
3. Initialize the word dictionary with the initial tokens (letters of the alphabet)
4. Run BPE algorithm (learn vocabulary)
5. Test token decomposition on selected sentences (apply learned rules)


In the documents you will observe a lot of TODO in the code, replace it by your code.


### The class we will complete


At the end we will create an object (python) class as it follows :

```python
class Tokenizer:
    ''' 
        A class for our Tokenizer

        Methods
        -------
        fit(text_corpus: str) : None
            Fit the tokenizer based on the input text corpus 
        tokenize(text: str): List[str]
            Tokenize a text and return the list of token ids
        detokenize(tokens: List[str]): str
            From a list of token ids return the corresponding text
    '''

    def __init__(self, vocabulary_size: int = 500):
        ''' 
            Parameters
            ----------
            vocabulary_size : int
                The expected size of the vocabulary

        '''
        super().__init__()
        self.vs = vocabulary_size

    def fit(self, text_corpus: str):
        ''' Train the tokenizer on the provided text.

            Parameters
            ----------
            text_corpus : str
                The text used to train the model
        '''
        raise NotImplementedError


    def tokenize(self, text: str) -> List[str]:
        ''' Tokenize a text.

            Parameters
            ----------
            text : str
                The text to tokenize

            Returns
            -------
            List[str]
                The list of tokens
        '''
        raise NotImplementedError

    def detokenize(self, tokens : List[int]) -> str:
        ''' Reverse the tokenization.

            Parameters
            ----------
            tokens : List[str]
                A list of token ids

            Returns
            -------
            str
                The text corresponding to tokens
        '''
        raise NotImplementedError

```

## Requirement

**For this lab you only need the python standard library :D**

In [166]:
import re # the regex library
import json # read export json format

from typing import List # to specify the types in function def
from collections import Counter # a tools to counts unique occurences

from urllib.request import urlopen, Request

### Step 1: Download a corpus

For this lab we will consider a wikipedia page in french [Grèce antique](https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique), but you are free to choose any content you want !

In [167]:


url_request  = 'https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique'
wikipedia_request = Request(url_request)
wikipedia_request.add_header("User-Agent", "Course (thomas.gerald@lisn.fr)")
raw_page = urlopen(wikipedia_request)
json_page = json.load(raw_page)


In [168]:
corpus = list(json_page['query']['pages'].values())[0]['extract']
print( corpus)

La Grèce antique est une civilisation de l'Antiquité des peuples de langue et de culture grecques développée en Grèce et dans la partie occidentale de l'Asie Mineure, puis, à la suite de plusieurs phases d'expansion, à Chypre, sur le pourtour de la mer Noire, en Sicile, en Italie du sud, en Cyrénaïque, en Égypte, au Levant méridional, en Syrie, constituant des points d'implantation jusque dans les actuelles Espagne et France à l'ouest, et jusqu'au territoire de l’actuel Afghanistan (Bactriane) à l'est.
Cette civilisation de culture grecque prend forme durant les « siècles obscurs » (v. 1200-800 av. J.-C.), à partir des décombres de la civilisation mycénienne, et se développe en particulier durant l'époque archaïque (v. 800-480 av. J.-C.), et s'épanouit pleinement durant l'époque classique (480-323 av. J.-C.) et l'époque hellénistique (323-31 av. J.-C.). La conquête romaine (entre 220 et 31 av. J.-C.) marque la fin de l'indépendance politique grecque, mais la culture grecque antique a c

### Step 2: Splitting text into words (or sequence of character no containing space)

To split the text into words, we'll use the following regex ```r'(\b[^\s]+\b)'```. To count words, we'll use python's Counter object. 
1. Store each word and its number of occurrences in **count_words**.
2. Give the 10 most frequent words (you'll store them in most_commons_words).

In [169]:

word_regex = re.compile(r'(\b[^\s]+\b)')
words = word_regex.findall(corpus)

print(words)
# count_words is a dictionary
count_words = {}
for w in words :
    if w in count_words:
        count_words[w]+=1
    else :
        count_words[w] = 1
# dict_items([
#     ("the", 1234),
#     ("of", 978),.
#     ("and", 856),
#     ...
# ])
# key=lambda x: x[1] compare element. x = ("the", 1234)        
most_commons_words = sorted(count_words.items(),
    key=lambda x: x[1], reverse=True)[: 10]

print(most_commons_words)

['La', 'Grèce', 'antique', 'est', 'une', 'civilisation', 'de', "l'Antiquité", 'des', 'peuples', 'de', 'langue', 'et', 'de', 'culture', 'grecques', 'développée', 'en', 'Grèce', 'et', 'dans', 'la', 'partie', 'occidentale', 'de', "l'Asie", 'Mineure', 'puis', 'à', 'la', 'suite', 'de', 'plusieurs', 'phases', "d'expansion", 'à', 'Chypre', 'sur', 'le', 'pourtour', 'de', 'la', 'mer', 'Noire', 'en', 'Sicile', 'en', 'Italie', 'du', 'sud', 'en', 'Cyrénaïque', 'en', 'Égypte', 'au', 'Levant', 'méridional', 'en', 'Syrie', 'constituant', 'des', 'points', "d'implantation", 'jusque', 'dans', 'les', 'actuelles', 'Espagne', 'et', 'France', 'à', "l'ouest", 'et', "jusqu'au", 'territoire', 'de', 'l’actuel', 'Afghanistan', 'Bactriane', 'à', "l'est", 'Cette', 'civilisation', 'de', 'culture', 'grecque', 'prend', 'forme', 'durant', 'les', 'siècles', 'obscurs', 'v', '1200-800', 'av', 'J.-C', 'à', 'partir', 'des', 'décombres', 'de', 'la', 'civilisation', 'mycénienne', 'et', 'se', 'développe', 'en', 'particulier',

### Step 3: Initialize word dictionary with initial tokens (letters of the alphabet)

Create the initial vocabulary in the vocab variable. How many initial tokens do you have?

In [170]:
# list
vocab = []
for w in count_words.keys():
    for c in w:
        if c not in vocab:
            vocab.append(c)
print(vocab)
print("number of initial tokens :", len(vocab))


['L', 'a', 'G', 'r', 'è', 'c', 'e', 'n', 't', 'i', 'q', 'u', 's', 'v', 'l', 'o', 'd', "'", 'A', 'é', 'p', 'g', 'M', 'à', 'h', 'x', 'C', 'y', 'm', 'N', 'S', 'I', 'ï', 'É', 'j', 'E', 'F', '’', 'f', 'B', 'b', '1', '2', '0', '-', '8', 'J', '.', '4', '3', 'ê', 'z', 'V', 'À', 'ù', 'O', 'D', 'ô', 'R', '5', '7', 'X', 'ç', 'Γ', 'ρ', 'α', 'ι', 'κ', 'ό', 'ς', 'k', 'ó', 'P', 'H', 'î', 'T', 'â', '6', '/', 'Â', 'Q', '(', 'K', 'Z', 'U', 'w', '9', 'œ', ')', '²', ',', 'û', 'W', 'Œ', 'Α', 'Ε', 'Ι', 'Ο', 'Υ', 'Ω', 'Φ', 'Χ', 'Ψ', 'ü', 'Y']
number of initial tokens : 105


### Step 4: Learning the tokenizer
To learn the tokenizer we need several functions:
1. A function to calculate the frequency of each token pair.
2. A function to merge a pair

Several variables will be required:
1. **vocab** containing current vocabulary
2. **merge_rules** containing all the merge rules (a dictionary containing as key a pair of tokens to merge and the result of the token merge). For example: {('e', 's'), 'es', ('en', 't') :'ent'}.
3. **splits** A dictionary containing the current breakdown of the corpus, with the word as key and the list of “tokens” as value.


In [171]:
# In the first step splits contains the words broken down into characters
splits = {}
for w in count_words.keys():
    splits[w] = []
    for c in w:
        splits[w].append(c)

print (splits)       



{'La': ['L', 'a'], 'Grèce': ['G', 'r', 'è', 'c', 'e'], 'antique': ['a', 'n', 't', 'i', 'q', 'u', 'e'], 'est': ['e', 's', 't'], 'une': ['u', 'n', 'e'], 'civilisation': ['c', 'i', 'v', 'i', 'l', 'i', 's', 'a', 't', 'i', 'o', 'n'], 'de': ['d', 'e'], "l'Antiquité": ['l', "'", 'A', 'n', 't', 'i', 'q', 'u', 'i', 't', 'é'], 'des': ['d', 'e', 's'], 'peuples': ['p', 'e', 'u', 'p', 'l', 'e', 's'], 'langue': ['l', 'a', 'n', 'g', 'u', 'e'], 'et': ['e', 't'], 'culture': ['c', 'u', 'l', 't', 'u', 'r', 'e'], 'grecques': ['g', 'r', 'e', 'c', 'q', 'u', 'e', 's'], 'développée': ['d', 'é', 'v', 'e', 'l', 'o', 'p', 'p', 'é', 'e'], 'en': ['e', 'n'], 'dans': ['d', 'a', 'n', 's'], 'la': ['l', 'a'], 'partie': ['p', 'a', 'r', 't', 'i', 'e'], 'occidentale': ['o', 'c', 'c', 'i', 'd', 'e', 'n', 't', 'a', 'l', 'e'], "l'Asie": ['l', "'", 'A', 's', 'i', 'e'], 'Mineure': ['M', 'i', 'n', 'e', 'u', 'r', 'e'], 'puis': ['p', 'u', 'i', 's'], 'à': ['à'], 'suite': ['s', 'u', 'i', 't', 'e'], 'plusieurs': ['p', 'l', 'u', 's',

#### Compute the frequency of token pairs
Create a function **compute_pair_freqs**, which, given the words broken down into tokens (splits dictionary) and the frequency of the words, returns the frequency of each pair of tokens (note only successive sub-words).

In [172]:
def compute_pair_freqs(splits, word_freqs):
    pair_freqs = {}
    # key value
    for word, freq in word_freqs.items():
        # letter of alphabet
        tokens = splits[word]  
        # pair
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            if pair in pair_freqs:
                pair_freqs[pair] += freq
            if pair not in pair_freqs:
                pair_freqs[pair] = freq

    return pair_freqs

In [173]:
pair_freqs = compute_pair_freqs(splits, count_words)
{k: pair_freqs[k] for k  in list(pair_freqs.keys())[:5]}

{('L', 'a'): 165,
 ('G', 'r'): 255,
 ('r', 'è'): 244,
 ('è', 'c'): 270,
 ('c', 'e'): 1163}

#### Find the most frequent pair and merge a pair
1. Create a function **most_frequent(pair_freqs)** returning the most frequent pair of tokens.
2. Create a **merge_pair()** function which, given a pair, returns the new splits of the corpus.

In [174]:
def most_frequent(pair_freqs):
    most_frequent = sorted(pair_freqs.items(), key=lambda x:x[1], reverse= True)[:1]
    return most_frequent

most_frequent(pair_freqs)

[(('e', 's'), 5663)]

In [175]:
from heapq import merge


def merge_pair(a : str, b: str, splits):
    '''
        splits : the dataset of words tokenized
        a : the first token  
        b : the second tokens

        return : splits with a,b tokens merged
    '''
    for word in splits.keys():
        token = splits[word]
        # Traversing backward, deletion does not affect indexes that have not yet been traversed.
        for i in range(len(token) - 2, -1, -1):
            if token[i] == a and token[i + 1] == b:
                token[i] = a + b
                del token[i + 1]
    return splits

In [176]:
new_splits = merge_pair(*most_frequent(pair_freqs)[0], splits)
print(new_splits['grecques'])

['g', 'r', 'e', 'c', 'q', 'u', 'e', 's']


#### Apply the algorithm until the desired vocabulary size is reached.
Create a BPE object that takes as arguments a corpus, a vocabulary size and train the BPE algorithm. The algorithm stores the final vocabulary in the **vocabulary** attribute and the merge rules in **merge_rules**.
For merge_rules, here's an example of its contents:
```
{('e', 's'): 'es',
 ('n', 't'): 'nt',
 ('q', 'u'): 'qu',
 ('r', 'e'): 're',
 ('o', 'n'): 'on',
 ('d', 'e'): 'de',
 ('l', 'e'): 'le',
 ('t', 'i'): 'ti',
 ('l', 'a'): 'la',
 ('i', 's'): 'is',
 ('e', 'nt'): 'ent', ...
 }
```

In [177]:
class BPE:
    def __init__(self, corpus, vocabulary_size=500):
        super().__init__()
        self.word_regex = re.compile(r'(\b[^\s]+\b)')
        words = self.word_regex.findall(corpus)

        # counting words
        count_words = Counter(words)
        # create initial vocab
        self.vocab = list({char for word in count_words.keys() for char in word })
        self.vocab.sort()
        # create the initial split
        splits = {word: [c for c in word] for word in count_words.keys()}
        # initialise merge_rules
        self.merge_rules = {}
        while len(self.vocab) < vocabulary_size:
            # compute pair frequencies
            pair_freqs = compute_pair_freqs(splits, count_words)
            if not pair_freqs:
                break

            # get best pair (most_frequent returns [(pair, freq)])
            (a, b), _ = most_frequent(pair_freqs)[0]

            # merge the pair in splits
            splits = merge_pair(a, b, splits)

            # record merge rule
            merged = a + b
            self.merge_rules[(a, b)] = merged

            # add new token to vocab
            if merged not in self.vocab:
                self.vocab.append(merged)

    
    def tokenize(self, text):
        words = self.word_regex.findall(text)
        splits = [[l for l in word] for word in words]
        for pair, merge in self.merge_rules.items():
            for idx, split in enumerate(splits):
                i = 0
                while i < len(split) - 1:
                    if split[i] == pair[0] and split[i + 1] == pair[1]:
                        split = split[:i] + [merge] + split[i + 2 :]
                    else:
                        i += 1
                splits[idx] = split
    
        return sum(splits, [])


In [178]:
my_bpe = BPE(corpus)

In [179]:
texte = '''culture grecques développée en Grèce '''
my_bpe.tokenize(texte)[:12]

['culture', 'grecques', 'dévelop', 'p', 'ée', 'en', 'Grèce']

#### Test by modifying parameters or corpus
Test the algorithm with different hyper-parameters or data

In [180]:
texte = '''Au plus large, l'Antiquité grecque s'étend de l'époque des palais minoens '''
my_bpe.tokenize(texte)[:10]

['A', 'u', 'plus', 'la', 'r', 'ge', "l'A", 'nti', 'qu', 'ité']

In [181]:
texte = '''l'identité locale, notamment celle construite autour de la cité (« politique ») '''
my_bpe.tokenize(texte)[:10]

["l'", 'i', 'dent', 'ité', 'lo', 'c', 'ale', 'notamment', 'ce', 'lle']

## Using sentencepiece
We're now going to use the `sentencepiece` library, which can be installed (if not already installed) with pip : 

`! pip install sentencepiece`

To train the tokenizer, we'll use the `train` function of `SentencePieceTrainer`. 

In [182]:
import sentencepiece as spm

with open("test-cours.txt", "w", encoding="utf-8") as f:
    f.write(corpus)
spm.SentencePieceTrainer.train(input="test-cours.txt", model_type='BPE',  model_prefix='m', vocab_size=500)

Looking at the “m.vocab” file, what are the differences with the vocabulary learned with your implementation? What modifications could you consider to obtain a similar vocabulary?

## Differences between SentencePiece vocabulary and our BPE implementation

By inspecting the `m.vocab` file produced by SentencePiece and comparing it with the vocabulary learned by our own BPE implementation, several key differences can be observed:

### 1. Presence of special tokens
SentencePiece includes special tokens such as `<unk>`, `<s>`, and `</s>` to handle unknown tokens and sentence boundaries.  
Our implementation does not include such tokens and therefore cannot explicitly represent unknown words or sentence delimiters.

### 2. Explicit modeling of word boundaries
SentencePiece encodes word boundaries explicitly using a special marker (represented here as `_`).  
As a result, tokens like `_la`, `_de`, or `_qu` appear in the vocabulary, allowing the model to distinguish between word-initial and word-internal subwords.  
In contrast, our implementation ignores whitespace and learns subwords only within pre-tokenized words.

### 3. More linguistically meaningful subwords
The SentencePiece vocabulary contains many common French morphemes and affixes such as `tion`, `ment`, `ent`, `ant`, and `que`.  
Our BPE implementation tends to produce more character-level or mechanically merged subwords, as it relies purely on frequency without additional linguistic constraints.

### 4. Richer handling of prefixes and suffixes
SentencePiece learns frequent prefixes and suffixes (e.g. `_re`, `_un`, `_en`) as independent subword units.  
Our implementation does not explicitly encourage such patterns and only merges pairs based on raw frequency.

### 5. Training on raw text instead of pre-tokenized words
SentencePiece operates directly on raw text without requiring prior word tokenization, which leads to a more flexible and robust subword vocabulary.  
Our implementation first splits the corpus into words and then applies BPE, which limits the diversity of learned subwords.

## Possible modifications to obtain a similar vocabulary

To obtain a vocabulary closer to that learned by SentencePiece, the following modifications could be considered:

- Explicitly model whitespace or word boundaries as special characters.
- Add special tokens such as `<unk>`, `<s>`, and `</s>` to the initial vocabulary.
- Apply frequency thresholds to avoid merging very rare token pairs.
- Avoid pre-tokenization and apply BPE directly on raw text.

Overall, SentencePiece produces a richer and more robust vocabulary by modeling word boundaries and special tokens explicitly, whereas our implementation is a simplified educational version of the BPE algorithm.

## Detokenization

Propose and implement a methods to detokenize encoded sentence (you should want to extend your vocabulary)

In our BPE implementation, tokens are concatenated subwords (e.g. `es`, `qu`, `tion`) and characters. Since tokenization is performed at the character/subword level and whitespace is not explicitly modeled, detokenization can be achieved by simply concatenating the tokens in order.

To make detokenization more robust and closer to realistic tokenizers, the vocabulary can be extended to explicitly represent word boundaries (e.g. using a special marker such as `_` for spaces). In that case, detokenization consists of concatenating tokens and then replacing the boundary markers by actual spaces.



In [ ]:
def detokenize(tokens):
    """
    Reconstruct a string from a list of BPE tokens.
    """
    return "".join(tokens)

This simple approach works because BPE tokens are learned as concatenations of characters. More advanced detokenization strategies could handle punctuation spacing, special tokens (such as <s> or </s>), and normalization rules to further improve text readability.